## Data Exploration & Understanding

### Load and Inspect

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/global_air_pollution_dataset.csv')

# Display the first few rows
display(df.head())

### Understand the Columns

In [ ]:
# Display dataset information including column names and data types
display(df.info())

### Summary Statistics

In [ ]:
# Generate summary statistics for numerical columns
display(df.describe())

### Check Data Types

In [ ]:
# Display the data types of each column
display(df.dtypes)

## Data Cleaning & Preprocessing

### Handle Missing Values

In [ ]:
# Identify columns with missing values
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
display(missing_values.sort_values(ascending=False))

In [ ]:
# The missing value imputation for 'Pollution_Alert_Level' and 'Associated_Diseases' has been moved to cell ca214cee
# to ensure it's applied after the dataset reload and before other feature engineering steps.

### Check for Duplicates

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# If duplicates exist, remove them
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
    print(f"New number of rows: {len(df)}")

### Handle Invalid Data

In [ ]:
# Check unique values for categorical columns to identify invalid entries
categorical_cols = df.select_dtypes(include='object').columns
print("Unique values for categorical columns:")
for col in categorical_cols:
    print(f"\nColumn '{col}':")
    print(df[col].value_counts())
    # Optionally, if too many unique values, print top N or sample
    # if len(df[col].unique()) > 50:
    #     print(df[col].value_counts().head())
    # else:
    #     print(df[col].value_counts())

### Handle Outliers

In [ ]:
# Identify numerical columns for outlier detection
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Exclude Year, Month, Latitude, Longitude, and WHO_PM25_Guideline_ugm3 from outlier treatment as they are often fixed or represent geographic coordinates.
# Also exclude 'Record_ID' if it was numerical, but it's an object type.
columns_to_exclude = ['Year', 'Month', 'Latitude', 'Longitude', 'WHO_PM25_Guideline_ugm3']
outlier_cols = [col for col in numerical_cols if col not in columns_to_exclude]

print("Columns considered for outlier treatment:", outlier_cols)

# Using IQR method to detect and cap outliers
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap outliers: values below lower_bound are set to lower_bound, values above upper_bound are set to upper_bound
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    print(f"Outliers in '{col}' capped between {lower_bound:.2f} and {upper_bound:.2f}")

# Re-check summary statistics to see the effect of outlier capping
display(df[outlier_cols].describe())

## EDA

### Univariate Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# List of features for univariate analysis
features_to_plot = ['AQI', 'PM2_5_ugm3', 'PM10_ugm3', 'NO2_ugm3', 'CO_mgm3']

print("Histograms for selected features:")
# Plot histograms
plt.figure(figsize=(15, 10))
for i, col in enumerate(features_to_plot):
    plt.subplot(2, 3, i + 1) # Adjust subplot grid based on number of features
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
print("Box plots for selected features:")
# Plot box plots
plt.figure(figsize=(15, 10))
for i, col in enumerate(features_to_plot):
    plt.subplot(2, 3, i + 1) # Adjust subplot grid based on number of features
    sns.boxplot(y=df[col])
    plt.title(f'Box Plot of {col}')
    plt.ylabel(col)
plt.tight_layout()
plt.show()

### Bivariate Analysis

In [ ]:
print("Scatter plots for selected feature relationships:")

# Plotting AQI against key pollutants
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.scatterplot(x='PM2_5_ugm3', y='AQI', data=df)
plt.title('AQI vs. PM2_5_ugm3')

plt.subplot(1, 3, 2)
sns.scatterplot(x='PM10_ugm3', y='AQI', data=df)
plt.title('AQI vs. PM10_ugm3')

plt.subplot(1, 3, 3)
sns.scatterplot(x='NO2_ugm3', y='AQI', data=df)
plt.title('AQI vs. NO2_ugm3')

plt.tight_layout()
plt.show()

In [ ]:
# Plotting a pollutant against a meteorological condition
plt.figure(figsize=(10, 5))
sns.scatterplot(x='Temperature_C', y='PM2_5_ugm3', data=df)
plt.title('PM2_5_ugm3 vs. Temperature_C')
plt.tight_layout()
plt.show()

### Time-Series Analysis

In [ ]:
# Convert 'Date' column to datetime objects if not already
df['Date'] = pd.to_datetime(df['Date'])

# Sort by date to ensure correct time series plotting
df_time_series = df.sort_values('Date')

# Set 'Date' as index for time series analysis
df_time_series = df_time_series.set_index('Date')

print("Time Series Plots for AQI and selected pollutants (2015-2024):")

# Plot AQI over time
plt.figure(figsize=(15, 6))
plt.plot(df_time_series['AQI'], label='AQI')
plt.title('AQI Over Time (2015-2024)')
plt.xlabel('Date')
plt.ylabel('AQI Value')
plt.legend()
plt.grid(True)
plt.show()

# Plot key pollutants over time
key_pollutants = ['PM2_5_ugm3', 'PM10_ugm3', 'NO2_ugm3', 'SO2_ugm3', 'O3_ugm3']
plt.figure(figsize=(15, 10))
for i, col in enumerate(key_pollutants):
    plt.subplot(len(key_pollutants), 1, i + 1)
    plt.plot(df_time_series[col], label=col, color=f'C{i}')
    plt.title(f'{col} Over Time')
    plt.xlabel('Date')
    plt.ylabel(col)
    plt.legend()
    plt.grid(True)
plt.tight_layout()
plt.show()

### Geographic Analysis

In [ ]:
# Analyze AQI by Country
aqi_by_country = df.groupby('Country')['AQI'].mean().sort_values(ascending=False)
print("Top 10 Countries by Average AQI:")
display(aqi_by_country.head(10))

# Visualize top countries by AQI
plt.figure(figsize=(12, 6))
sns.barplot(x=aqi_by_country.head(10).index, y=aqi_by_country.head(10).values)
plt.title('Top 10 Countries by Average AQI')
plt.xlabel('Country')
plt.ylabel('Average AQI')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze PM2.5 by City (example for a pollutant)
pm25_by_city = df.groupby('City')['PM2_5_ugm3'].mean().sort_values(ascending=False)
print("Top 10 Cities by Average PM2.5:")
display(pm25_by_city.head(10))

# Visualize top cities by PM2.5
plt.figure(figsize=(12, 6))
sns.barplot(x=pm25_by_city.head(10).index, y=pm25_by_city.head(10).values)
plt.title('Top 10 Cities by Average PM2.5')
plt.xlabel('City')
plt.ylabel('Average PM2.5 (ug/m3)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Correlation Analysis

In [ ]:
# Select numerical columns for correlation analysis
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Exclude ID and Date related numerical columns if they don't represent continuous measures
# 'Year', 'Month' are often treated as categorical or temporal index in some analyses.
# 'WHO_PM25_Guideline_ugm3' is a constant, so it won't have correlation.

correlation_features = [col for col in numerical_cols if col not in ['Year', 'Month', 'WHO_PM25_Guideline_ugm3']]

# Compute the correlation matrix
correlation_matrix = df[correlation_features].corr()

# Plot the correlation matrix as a heatmap
plt.figure(figsize=(20, 18))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

In [ ]:
# Display correlations with the target variable (AQI) as a sorted list
print("Correlations with AQI:")
display(correlation_matrix['AQI'].sort_values(ascending=False))

## Feature Engineering

### Encode Categorical Values

In [ ]:
# --- Ensure df is in a consistent state for Feature Engineering ---# Reload the dataset to get a fresh start for feature engineeringdf = pd.read_csv('/content/global_air_pollution_dataset.csv')# Convert 'Date' column to datetime objects (needed for sorting and date feature extraction later)df['Date'] = pd.to_datetime(df['Date'])# Re-apply Missing Value Imputation (from cell b384452c)for col in ['Pollution_Alert_Level', 'Associated_Diseases']:    if col in df.columns and df[col].isnull().any():        df[col].fillna(df[col].mode()[0], inplace=True)# Re-apply Outlier Capping (from cell 8ba59ca2)numerical_cols = df.select_dtypes(include=['int64', 'float64']).columnscolumns_to_exclude = ['Year', 'Month', 'Latitude', 'Longitude', 'WHO_PM25_Guideline_ugm3']outlier_cols = [col for col in numerical_cols if col not in columns_to_exclude]for col in outlier_cols:    Q1 = df[col].quantile(0.25)    Q3 = df[col].quantile(0.75)    IQR = Q3 - Q1    lower_bound = Q1 - 1.5 * IQR    upper_bound = Q3 + 1.5 * IQR    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)# --- Drop Leaky and Constant Features EARLY to prevent target leakage ---leaky_and_constant_features_to_drop = [    'AQI_Category', 'Pollution_Alert_Level', 'Health_Effects', 'Associated_Diseases',    'Daily_Mortality_Risk', 'Exceeds_WHO_PM25_Standard', 'WHO_PM25_Guideline_ugm3',    'Respiratory_Risk_Index', 'Cardiovascular_Risk_Index', 'Cancer_Risk_Index' # Added these three risk indices]# --- DEBUG PRINTS ---print("\n--- Debugging df.drop operation ---")print(f"Columns in df BEFORE drop: {list(df.columns)}")# Filter to ensure only columns present in the DataFrame are droppedleaky_and_constant_features_to_drop_actual = [col for col in leaky_and_constant_features_to_drop if col in df.columns]print(f"Columns ACTUALLY identified for dropping: {leaky_and_constant_features_to_drop_actual}")if leaky_and_constant_features_to_drop_actual:    # Use explicit assignment instead of inplace=True for robustness    df = df.drop(columns=leaky_and_constant_features_to_drop_actual)    print(f"Columns in df AFTER drop: {list(df.columns)}")else:    print("No leaky/constant features found in df to drop.")print("--- End Debug ---\n")print("DataFrame prepared for Feature Engineering: Missing values imputed and outliers capped.")print("Leaky and constant features dropped to prevent target leakage.")print(f"Initial shape for Feature Engineering: {df.shape}")

In [ ]:
import pandas as pd

# Ordinal Encoding for 'Traffic_Density'
traffic_density_mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Very High': 3}
df['Traffic_Density_Encoded'] = df['Traffic_Density'].map(traffic_density_mapping)

# Daily_Mortality_Risk is considered a leaky feature and will be dropped, so no ordinal encoding here.

# Identify nominal categorical columns for One-Hot Encoding
# Exclude leaky features (AQI_Category, Pollution_Alert_Level, Health_Effects, Associated_Diseases, Exceeds_WHO_PM25_Standard)
# Exclude high cardinality columns (Country, City, Primary_Pollution_Source, Record_ID)
initial_nominal_cols = [
    'Season', 'Wind_Direction', 'Weather_Condition', 'Industry_Nearby'
]
# Filter to ensure we only try to encode columns that still exist in the DataFrame
nominal_cols_to_encode = [col for col in initial_nominal_cols if col in df.columns]

# Perform One-Hot Encoding
df = pd.get_dummies(df, columns=nominal_cols_to_encode, drop_first=True)

# Display the first few rows with new encoded columns and check shapes
display(df[['Traffic_Density', 'Traffic_Density_Encoded']].head())
print(f"Shape after encoding: {df.shape}")

### Scale Numerical Features

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns to scale (excluding encoded, date, ID, target, and constant columns)
# Exclude 'Year', 'Month', 'WHO_PM25_Guideline_ugm3' as before.
# Exclude 'AQI' if it's the target variable (will be handled later).
# Exclude 'Record_ID' and 'Date' as they are not features to be scaled.

# Original numerical columns (from df.info() and df.describe() previously)
original_numerical_cols = [
    'Population_Density_per_km2', 'Green_Cover_Pct', 'PM2_5_ugm3', 'PM10_ugm3',
    'NO2_ugm3', 'SO2_ugm3', 'CO_mgm3', 'O3_ugm3', 'VOC_ugm3', 'NH3_ugm3',
    'Lead_Pb_ugm3', 'Benzene_ugm3', 'Formaldehyde_ugm3', 'Black_Carbon_ugm3',
    'Temperature_C', 'Humidity_Pct', 'Wind_Speed_kmh', 'Atmospheric_Pressure_hPa',
    'Visibility_km', 'Respiratory_Risk_Index', 'Cardiovascular_Risk_Index', 'Cancer_Risk_Index'
]

# Filter to ensure only present columns are selected (some might be dropped or transformed)
features_to_scale = [col for col in original_numerical_cols if col in df.columns]

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print("Numerical features scaled:")
display(df[features_to_scale].head())

### Create Date-Based Features

In [ ]:
# Extract useful features from the 'Date' column
# The 'Date' column was already converted to datetime during time series analysis.

df['Day_of_Week'] = df['Date'].dt.dayofweek
df['Day_of_Year'] = df['Date'].dt.dayofyear
df['Week_of_Year'] = df['Date'].dt.isocalendar().week.astype(int)

# Display the first few rows with new date features
display(df[['Date', 'Year', 'Month', 'Day_of_Week', 'Day_of_Year', 'Week_of_Year']].head())
print(f"Shape after adding date features: {df.shape}")

### Feature Creation

In [ ]:
# Example: Create a new feature 'Pollution_Ratio_PM25_PM10'
df['Pollution_Ratio_PM25_PM10'] = df['PM2_5_ugm3'] / (df['PM10_ugm3'] + 1e-6) # Add a small constant to avoid division by zero

# Example: Create an interaction term between Temperature and Humidity
df['Temp_Humidity_Interaction'] = df['Temperature_C'] * df['Humidity_Pct']

# Display the first few rows with new created features
display(df[['PM2_5_ugm3', 'PM10_ugm3', 'Pollution_Ratio_PM25_PM10', 'Temperature_C', 'Humidity_Pct', 'Temp_Humidity_Interaction']].head())
print(f"Shape after creating new features: {df.shape}")

## Model Building (Ensemble Learning)

### Define the Target Variable

In [ ]:
# Define target variable (y) and features (X)
y = df['AQI']

# List of columns to explicitly exclude from the feature set X to prevent target leakage and remove identifiers:
# This comprehensive list includes target, identifiers, high-cardinality nominals not encoded,
# original leaky features, constant features, and their one-hot encoded versions.
columns_to_exclude_from_X = [
    'AQI', 'Record_ID', 'Date',
    'Country', 'City', 'Primary_Pollution_Source',
    'Traffic_Density', # Original string column, replaced by 'Traffic_Density_Encoded'

    # Original leaky and constant features
    'AQI_Category', 'Pollution_Alert_Level', 'Health_Effects', 'Associated_Diseases',
    'Daily_Mortality_Risk', 'Exceeds_WHO_PM25_Standard', 'WHO_PM25_Guideline_ugm3',
    'Respiratory_Risk_Index', 'Cardiovascular_Risk_Index', 'Cancer_Risk_Index',

    # One-hot encoded versions of leaky categorical features (as observed in X_train/X_test)
    'AQI_Category_Hazardous',
    'AQI_Category_Moderate',
    'AQI_Category_Unhealthy',
    'AQI_Category_Unhealthy_for_Sensitive_Groups',
    'AQI_Category_Very_Unhealthy',
    'Pollution_Alert_Level_Emergency',
    'Pollution_Alert_Level_Hazardous_Emergency',
    'Pollution_Alert_Level_Warning',
    'Pollution_Alert_Level_Watch',
    'Health_Effects_Everyone_may_experience_health_effects_sensitive_groups_may_experience_serious_effects',
    'Health_Effects_Health_alert_serious_health_effects_for_everyone_emergency_conditions_for_sensitive_groups',
    'Health_Effects_Health_emergency_everyone_is_affected_life_threatening_for_sensitive_groups',
    'Health_Effects_No_significant_health_effects',
    'Health_Effects_Unusually_sensitive_people_may_experience_minor_respiratory_symptoms',
    'Associated_Diseases_COPD_Ischemic_Heart_Disease_Stroke_Risk_Lung_Irritation',
    'Associated_Diseases_Lung_Cancer_Risk_Heart_Attack_Arrhythmia_Premature_Death_Risk',
    'Associated_Diseases_Mild_asthma_exacerbation',
    'Associated_Diseases_Severe_COPD_Heart_Failure_Lung_Cancer_Neurological_Damage_Premature_Death',
    'Daily_Mortality_Risk_Encoded',
    'Exceeds_WHO_PM25_Standard_Yes'
]

# Filter out columns that might not exist in case of previous runs or changes
columns_to_exclude_from_X = [col for col in columns_to_exclude_from_X if col in df.columns]
X = df.drop(columns=columns_to_exclude_from_X, axis=1)

print("Target variable 'y' defined.")
print("Features 'X' defined, with all leaky features and identifiers explicitly removed.")
display(y.head())
display(X.head())

### Train/Test Split

In [ ]:
import pandas as pd

# Time-based split: train on 2015-2022, test on 2023-2024
# Ensure 'Date' column is datetime and sorted for time-based splitting
# df['Date'] = pd.to_datetime(df['Date']) # This is already done in ca214cee but keeping for safety.
df_sorted = df.sort_values(by='Date')

# Define training and testing periods
train_start_year = 2015
train_end_year = 2022
test_start_year = 2023
test_end_year = 2024

# Split data based on year
train_df = df_sorted[(df_sorted['Year'] >= train_start_year) & (df_sorted['Year'] <= train_end_year)]
test_df = df_sorted[(df_sorted['Year'] >= test_start_year) & (df_sorted['Year'] <= test_end_year)]

# Columns to explicitly drop from X_train and X_test to prevent target leakage and remove identifiers.
# This list should be consistent with `columns_to_exclude_from_X` defined when creating the global X.
columns_to_drop_for_modeling = [
    'AQI', 'Record_ID', 'Date',
    'Country', 'City', 'Primary_Pollution_Source',
    'Traffic_Density', # Original string column, replaced by 'Traffic_Density_Encoded'

    # Original leaky and constant features
    'AQI_Category', 'Pollution_Alert_Level', 'Health_Effects', 'Associated_Diseases',
    'Daily_Mortality_Risk', 'Exceeds_WHO_PM25_Standard', 'WHO_PM25_Guideline_ugm3',
    'Respiratory_Risk_Index', 'Cardiovascular_Risk_Index', 'Cancer_Risk_Index',

    # One-hot encoded versions of leaky categorical features (as observed in X_train/X_test)
    'AQI_Category_Hazardous',
    'AQI_Category_Moderate',
    'AQI_Category_Unhealthy',
    'AQI_Category_Unhealthy_for_Sensitive_Groups',
    'AQI_Category_Very_Unhealthy',
    'Pollution_Alert_Level_Emergency',
    'Pollution_Alert_Level_Hazardous_Emergency',
    'Pollution_Alert_Level_Warning',
    'Pollution_Alert_Level_Watch',
    'Health_Effects_Everyone_may_experience_health_effects_sensitive_groups_may_experience_serious_effects',
    'Health_Effects_Health_alert_serious_health_effects_for_everyone_emergency_conditions_for_sensitive_groups',
    'Health_Effects_Health_emergency_everyone_is_affected_life_threatening_for_sensitive_groups',
    'Health_Effects_No_significant_health_effects',
    'Health_Effects_Unusually_sensitive_people_may_experience_minor_respiratory_symptoms',
    'Associated_Diseases_COPD_Ischemic_Heart_Disease_Stroke_Risk_Lung_Irritation',
    'Associated_Diseases_Lung_Cancer_Risk_Heart_Attack_Arrhythmia_Premature_Death_Risk',
    'Associated_Diseases_Mild_asthma_exacerbation',
    'Associated_Diseases_Severe_COPD_Heart_Failure_Lung_Cancer_Neurological_Damage_Premature_Death',
    'Daily_Mortality_Risk_Encoded',
    'Exceeds_WHO_PM25_Standard_Yes'
]

# Filter out columns that might not exist before dropping
columns_to_drop_for_modeling = [col for col in columns_to_drop_for_modeling if col in train_df.columns]
X_train = train_df.drop(columns=columns_to_drop_for_modeling, axis=1)
y_train = train_df['AQI']

X_test = test_df.drop(columns=columns_to_drop_for_modeling, axis=1)
y_test = test_df['AQI']

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

In [ ]:
print("\nColumns in X_train:")
print(X_train.columns.tolist())
print("\nColumns in X_test:")
print(X_test.columns.tolist())

### Sanitize Feature Names for Model Compatibility

In [ ]:
import re

# Function to sanitize column names
def sanitize_names(df):
    cols = df.columns
    new_cols = []
    for col in cols:
        # Replace non-alphanumeric (and underscore) with single underscore
        new_col = re.sub(r'[\W_]+', '_', col)
        # Remove leading/trailing underscores
        new_col = new_col.strip('_')
        new_cols.append(new_col)
    df.columns = new_cols
    return df

# Apply sanitization to X_train and X_test to ensure consistent feature names
# We use .copy() to avoid SettingWithCopyWarning if these dataframes were views
X_train = sanitize_names(X_train.copy())
X_test = sanitize_names(X_test.copy())

print("Feature names in X_train and X_test sanitized for model compatibility.")
print(f"Sanitized X_train columns: {X_train.columns.tolist()[:5]}...")
print(f"Sanitized X_test columns: {X_test.columns.tolist()[:5]}...")

### Baseline model

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Initialize the Decision Tree Regressor
baseline_model = DecisionTreeRegressor(random_state=42)

# Train the baseline model on the training data
baseline_model.fit(X_train, y_train)

print("Baseline Decision Tree Regressor trained successfully.")

### Implement Bagging

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Initialize Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42)

# Define hyperparameters for tuning
param_grid_rf = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

# Use TimeSeriesSplit for cross-validation
tscv = TimeSeriesSplit(n_splits=3) # Using 3 splits for demonstration; increase for more robust tuning

# Set up GridSearchCV
grid_search_rf = GridSearchCV(estimator=rf_model, param_grid=param_grid_rf, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the training data
grid_search_rf.fit(X_train, y_train)

# Get the best parameters and best score
best_rf_params = grid_search_rf.best_params_
best_rf_score = grid_search_rf.best_score_

print(f"Best Random Forest parameters: {best_rf_params}")
print(f"Best Random Forest score (negative MSE): {best_rf_score}")

# Train the Random Forest model with the best parameters
final_rf_model = RandomForestRegressor(**best_rf_params, random_state=42)
final_rf_model.fit(X_train, y_train)

print("Random Forest Regressor with tuned hyperparameters trained successfully.")

### Implement Boosting

In [ ]:
import xgboost as xgb
from lightgbm import LGBMRegressor

# Initialize XGBoost Regressor
xgb_model = xgb.XGBRegressor(random_state=42, tree_method='hist')

# Define hyperparameters for tuning XGBoost
param_grid_xgb = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

# Use TimeSeriesSplit for cross-validation (re-using tscv from Random Forest)
# Set up GridSearchCV for XGBoost
grid_search_xgb = GridSearchCV(estimator=xgb_model, param_grid=param_grid_xgb, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the training data
grid_search_xgb.fit(X_train, y_train)

# Get the best parameters and best score
best_xgb_params = grid_search_xgb.best_params_
best_xgb_score = grid_search_xgb.best_score_

print(f"Best XGBoost parameters: {best_xgb_params}")
print(f"Best XGBoost score (negative MSE): {best_xgb_score}")

# Train the XGBoost model with the best parameters
final_xgb_model = xgb.XGBRegressor(**best_xgb_params, random_state=42, tree_method='hist')
final_xgb_model.fit(X_train, y_train)

print("XGBoost Regressor with tuned hyperparameters trained successfully.")

In [ ]:
import re

# Function to sanitize column names for LightGBM
def sanitize_names(df):
    cols = df.columns
    new_cols = []
    for col in cols:
        new_col = re.sub(r'[\W_]+', '_', col) # Replace non-alphanumeric (and underscore) with single underscore
        new_col = new_col.strip('_') # Remove leading/trailing underscores
        new_cols.append(new_col)
    df.columns = new_cols
    return df

# Apply sanitization to X_train and X_test
X_train = sanitize_names(X_train.copy())
X_test = sanitize_names(X_test.copy())

# Initialize LightGBM Regressor
lgbm_model = LGBMRegressor(random_state=42, verbose=-1) # verbose=-1 to suppress verbose output

# Define hyperparameters for tuning LightGBM
param_grid_lgbm = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

# Set up GridSearchCV for LightGBM
grid_search_lgbm = GridSearchCV(estimator=lgbm_model, param_grid=param_grid_lgbm, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the training data
grid_search_lgbm.fit(X_train, y_train)

# Get the best parameters and best score
best_lgbm_params = grid_search_lgbm.best_params_
best_lgbm_score = grid_search_lgbm.best_score_

print(f"Best LightGBM parameters: {best_lgbm_params}")
print(f"Best LightGBM score (negative MSE): {best_lgbm_score}")

# Train the LightGBM model with the best parameters
final_lgbm_model = LGBMRegressor(**best_lgbm_params, random_state=42, verbose=-1)
final_lgbm_model.fit(X_train, y_train)

print("LightGBM Regressor with tuned hyperparameters trained successfully.")

### Implement Stacking

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit, KFold # Import KFold

# Define base learners using the best trained models
estimators = [
    ('rf', final_rf_model),
    ('xgb', final_xgb_model),
    ('lgbm', final_lgbm_model)
]

# Define the meta-learner (final_estimator)
meta_learner = LinearRegression()

# The ValueError "cross_val_predict only works for partitions" occurs because TimeSeriesSplit
# does not ensure that every sample in the training data is included in a test fold.
# Specifically, the initial samples are only ever used for training, not testing, in TimeSeriesSplit.
# StackingRegressor's internal cross_val_predict expects every sample to appear in exactly one test set.
# To resolve this, we will use a standard KFold for the internal cross-validation of StackingRegressor.
# The overall time-based train/test split (2015-2022 vs 2023-2024) still ensures temporal separation.

# Use KFold for cross-validation within StackingRegressor
# We can use the same number of splits as we intended for TimeSeriesSplit.
kf_stack = KFold(n_splits=5, shuffle=False) # shuffle=False maintains data order if it's already sorted by time

# Build the Stacking Regressor
stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=meta_learner,
    cv=kf_stack, # Use KFold here instead of tscv_stack
    n_jobs=-1, # Use all available cores
    verbose=1
)

# Train the stacking model
stacking_model.fit(X_train, y_train)

print("Stacking Regressor trained successfully.")

### Download the model

In [ ]:
import joblib

# Save the best ensemble model (e.g., the stacking model) as a .pkl file
model_filename = 'stacking_regressor_model.pkl'
joblib.dump(stacking_model, model_filename)

print(f"Best model saved as {model_filename}")

## Model Evaluation & Comparison

After training all our models, it's essential to evaluate their performance on the test set and compare them to determine the best-performing model for our air quality prediction task.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Dictionary to store model predictions and evaluation results
predictions = {}
results = {}

# --- 1. Evaluate Baseline Decision Tree Model ---
# Make predictions on the test set
y_pred_baseline = baseline_model.predict(X_test)
predictions['Baseline_DT'] = y_pred_baseline

# Calculate evaluation metrics
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mse_baseline)
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)

results['Baseline_DT'] = {'MSE': mse_baseline, 'RMSE': rmse_baseline, 'MAE': mae_baseline}

print("Baseline Decision Tree Evaluation:")
print(f"  MSE: {mse_baseline:.4f}")
print(f"  RMSE: {rmse_baseline:.4f}")
print(f"  MAE: {mae_baseline:.4f}")
print("\n" + "-"*50 + "\n")

# --- 2. Evaluate Random Forest Model ---
y_pred_rf = final_rf_model.predict(X_test)
predictions['Random_Forest'] = y_pred_rf

mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

results['Random_Forest'] = {'MSE': mse_rf, 'RMSE': rmse_rf, 'MAE': mae_rf}

print("Random Forest Evaluation:")
print(f"  MSE: {mse_rf:.4f}")
print(f"  RMSE: {rmse_rf:.4f}")
print(f"  MAE: {mae_rf:.4f}")
print("\n" + "-"*50 + "\n")

# --- 3. Evaluate XGBoost Model ---
y_pred_xgb = final_xgb_model.predict(X_test)
predictions['XGBoost'] = y_pred_xgb

mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

results['XGBoost'] = {'MSE': mse_xgb, 'RMSE': rmse_xgb, 'MAE': mae_xgb}

print("XGBoost Evaluation:")
print(f"  MSE: {mse_xgb:.4f}")
print(f"  RMSE: {rmse_xgb:.4f}")
print(f"  MAE: {mae_xgb:.4f}")
print("\n" + "-"*50 + "\n")

# --- 4. Evaluate LightGBM Model ---
y_pred_lgbm = final_lgbm_model.predict(X_test)
predictions['LightGBM'] = y_pred_lgbm

mse_lgbm = mean_squared_error(y_test, y_pred_lgbm)
rmse_lgbm = np.sqrt(mse_lgbm)
mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)

results['LightGBM'] = {'MSE': mse_lgbm, 'RMSE': rmse_lgbm, 'MAE': mae_lgbm}

print("LightGBM Evaluation:")
print(f"  MSE: {mse_lgbm:.4f}")
print(f"  RMSE: {rmse_lgbm:.4f}")
print(f"  MAE: {mae_lgbm:.4f}")
print("\n" + "-"*50 + "\n")

# --- 5. Evaluate Stacking Regressor Model ---
y_pred_stack = stacking_model.predict(X_test)
predictions['Stacking_Regressor'] = y_pred_stack

mse_stack = mean_squared_error(y_test, y_pred_stack)
rmse_stack = np.sqrt(mse_stack)
mae_stack = mean_absolute_error(y_test, y_pred_stack)

results['Stacking_Regressor'] = {'MSE': mse_stack, 'RMSE': rmse_stack, 'MAE': mae_stack}

print("Stacking Regressor Evaluation:")
print(f"  MSE: {mse_stack:.4f}")
print(f"  RMSE: {rmse_stack:.4f}")
print(f"  MAE: {mae_stack:.4f}")
print("\n" + "-"*50 + "\n")

# Convert results to a DataFrame for easy comparison
results_df = pd.DataFrame(results).T
print("Model Comparison (Lower is better for all metrics):")
display(results_df.sort_values(by='RMSE'))

## Model Explainability & Ethics

Understanding *why* a model makes certain predictions is as important as its predictive accuracy, especially in sensitive domains like public health and environmental monitoring. This section focuses on explaining our best model's behavior and discussing ethical considerations.

### Global Explainability (SHAP)

SHAP (SHapley Additive exPlanations) is a game theory approach to explain the output of any machine learning model. It connects optimal credit allocation with local explanations by using Shapley values. Here, we'll use SHAP to understand the overall importance of features for our best model (Random Forest in this case, based on the previous evaluation).

In [ ]:
# Install shap if not already installed
# !pip install shap -q

import shap

# For tree-based models, shap.TreeExplainer is efficient
# We will use the Random Forest model as it performed the best based on RMSE
explainer = shap.TreeExplainer(final_rf_model)

# Calculate SHAP values for the test set
# This can be computationally intensive for large datasets
shap_values = explainer.shap_values(X_test)

print("SHAP values calculated.")

In [ ]:
print("SHAP Summary Plot (Global Feature Importance):")
# Plot the SHAP summary
# The summary plot shows which features are most important and how they impact the model's output.
shap.summary_plot(shap_values, X_test, plot_type="bar")

In [ ]:
print("SHAP Beeswarm Plot (Feature Impact and Direction):")
shap.summary_plot(shap_values, X_test)

### Local Explainability (SHAP for individual predictions)

Local explainability focuses on explaining why a model made a specific prediction for a single instance. This can be crucial for debugging, building trust, and understanding specific cases.

In [ ]:
# Choose a specific instance from the test set for local explanation
# Let's pick the first instance in the test set
instance_idx = 0
sample_instance = X_test.iloc[[instance_idx]]
sample_true_aqi = y_test.iloc[instance_idx]
sample_predicted_aqi = final_rf_model.predict(sample_instance)[0]

print(f"Explaining prediction for instance index: {instance_idx}")
print(f"True AQI: {sample_true_aqi:.2f}")
print(f"Predicted AQI: {sample_predicted_aqi:.2f}")

# Calculate SHAP values for the single instance
shap_values_instance = explainer.shap_values(sample_instance)

# Plot the SHAP force plot for this instance
# This plot shows how each feature contributes to the prediction for this specific instance.
print("SHAP Force Plot for a single instance:")
shap.initjs() # Initialize Javascript for rendering
shap.force_plot(explainer.expected_value, shap_values_instance, sample_instance)

### Discussion of Limitations & Ethics

Even with advanced models and explainability techniques, it's crucial to consider the limitations and ethical implications of our air pollution prediction model.

**Potential Biases:**
*   **Data Bias:** If the training data disproportionately represents certain regions, seasons, or pollution events, the model might perform poorly on underrepresented scenarios. For instance, if data from rural areas is scarce, the model's predictions there might be less accurate.
*   **Measurement Bias:** Inaccuracies or inconsistencies in pollution sensor data could propagate bias into the model's learning process.
*   **Feature Bias:** The selection of features, or how they are engineered, could inadvertently embed societal biases (e.g., if socio-economic indicators correlated with historical pollution are used, it might perpetuate historical injustices).

**Fairness Concerns:**
*   **Unequal Impact:** A model that works well overall might still perform poorly for specific communities or demographics, especially those historically marginalized or with unique environmental challenges. This could lead to unfair resource allocation or inaccurate health warnings.
*   **Predictive Parity:** The model's error rates (false positives/negatives) should ideally be consistent across different groups (e.g., different cities, income levels, or ethnic groups).

**Privacy Issues:**
*   While individual-level data isn't used here, location data (Latitude, Longitude, City, Country) could potentially be combined with other datasets to infer sensitive information about communities.
*   Ensuring anonymization and aggregation where necessary is important, especially if finer-grained predictions are made available to the public.

**Cost of False Positives vs. False Negatives:**
*   **False Positive (predicting high AQI when it's low):** Could lead to unnecessary public alerts, economic disruptions (e.g., closures), and reduced trust in the warning system over time.
*   **False Negative (predicting low AQI when it's high):** This is generally more critical in air quality prediction. It could result in significant public health risks, inadequate protective measures by individuals, and delayed policy responses, potentially leading to increased illness and mortality.
*   The specific application of the model dictates which type of error is more acceptable. For public health, minimizing false negatives is usually paramount.

**Need for Human Oversight:**
*   **Model Drift:** Air pollution patterns, meteorological conditions, and industrial activities change over time. Models require continuous monitoring and retraining to ensure their relevance and accuracy.
*   **Contextual Understanding:** Automated models lack the nuanced understanding of local events (e.g., sudden industrial accidents, large-scale wildfires, policy changes) that human experts possess. Human oversight is essential to interpret model outputs in context, especially during unusual events.
*   **Ethical Review:** Regular ethical reviews should be conducted to assess the model's impact on different communities, ensuring it aligns with public good and does not exacerbate existing inequalities.

In conclusion, while this machine learning project provides powerful tools for air pollution analysis, its real-world deployment must be accompanied by rigorous ethical considerations, continuous validation, and human expertise to ensure responsible and equitable outcomes.